In [ ]:
# Test cell - simple Python execution
print("Kernel is working!")
import sys
print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

In [ ]:
import pandas as pd
df = pd.read_csv("data/agragado_meteo_vazao_shifted_station_19091_extended.csv")
df.head()

In [ ]:
# =============================================================================
# v7.7_REFINED_SAFE_SHAP_GA2targetCOV88.py — Hybrid Hydrological Forecast System (HHFS)
# =============================================================================
# Author: Gutemberg Borges França (LMA-UFRJ)
# Station: Santa Branca Outlet (ID 19091)
# -----------------------------------------------------------------------------
# GA₁ — Deterministic NSE/KGE-optimized Forecast (meta-learning + seasonal calibration)
# GA₂ — Hydro-Adaptive Uncertainty Calibration (target COV ≈ 0.88)
# -----------------------------------------------------------------------------
# Refinements:
#   - Monthly mean & variance correction (improves KGE)
#   - Weighted training by flow magnitude (improves NSE)
#   - Dual Ridge meta-learner (wet/dry regimes)
#   - Robust GA₂ with target coverage 0.88 ±0.02
#   - SHAP importance (full 41 features)
# =============================================================================

!pip install shap xgboost lightgbm deap openpyxl matplotlib scikit-learn --quiet

import numpy as np, pandas as pd, shap, warnings, matplotlib.pyplot as plt, random, os
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from deap import base, creator, tools, algorithms
from scipy.stats import pearsonr
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")

OUT_PATH = "/content/drive/MyDrive/PREVISÃO-VAZAO-FUNIL"
os.makedirs(OUT_PATH, exist_ok=True)
print("\n===== Station 19091 | v7.7_REFINED_SAFE_SHAP_GA2targetCOV88 =====")

# =============================================================================
# 1 LOAD AND PREPROCESS DATA
# =============================================================================
print("\n[1/10] Loading dataset and computing climatology...")
path = f"{OUT_PATH}/agragado_meteo_vazao_shifted_station_19091_extended.csv"
df = pd.read_csv(path)
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df["year"], df["month"] = df["date"].dt.year, df["date"].dt.month

df["pr_mean"] = (df["pr_min"] + df["pr_max"]) / 2
clim_y = df.groupby("month")["flow_next_month"].mean()
df["y_anom"] = df["flow_next_month"] - df["month"].map(clim_y)

# =============================================================================
# 2️ FEATURE ENGINEERING + SAFE SHIFT (ajustado p/ GA₂)
# =============================================================================
print("[2/10] Generating lag features and rainfall memory...")

# --- Criação de defasagens (lags) de vazão e precipitação
for k in range(1, 7):
    df[f"y_lag{k}"] = df["flow_next_month"].shift(k)
    df[f"pr_lag{k}"] = df["pr_mean"].shift(k)

# --- Somas móveis e médias móveis de chuva e vazão
for w in [3, 4, 5, 6]:
    df[f"pr_sum{w}"] = df["pr_mean"].rolling(w, min_periods=1).sum().shift(1)
    df[f"y_rm{w}"] = df["flow_next_month"].rolling(w, min_periods=1).mean().shift(1)

# --- Índice de precipitação antecedente (API)
df["pr_api5"] = (
    0.4 * df["pr_mean"].shift(1)
    + 0.25 * df["pr_mean"].shift(2)
    + 0.2 * df["pr_mean"].shift(3)
    + 0.1 * df["pr_mean"].shift(4)
    + 0.05 * df["pr_mean"].shift(5)
)

# --- Mantém min/max para GA₂ (usado na calibração da incerteza)
df["flow_max_prev"] = df["flow_next_month_max"].shift(1)
df["flow_min_prev"] = df["flow_next_month_min"].shift(1)

# Importante:
# NÃO remover as colunas flow_next_month_min/max — elas serão usadas no GA₂
# para cálculo da cobertura (COV) e das bandas [low, up].
# df.drop(columns=["flow_next_month_max","flow_next_month_min"], inplace=True)  ← REMOVIDO

# --- Codificação harmônica do ciclo anual
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# --- Remoção de linhas iniciais com NaN (causadas por shift)
df.dropna(inplace=True)

# --- Lista completa de preditores
X_cols = [
    "month","u2_min","u2_max","tmin_min","tmin_max","tmax_min","tmax_max",
    "rs_min","rs_max","rh_min","rh_max","eto_min","eto_max","pr_min","pr_max",
    "pr_mean","flow_max_prev","flow_min_prev","y_lag1","pr_lag1","y_lag2","pr_lag2",
    "y_lag3","pr_lag3","y_lag4","pr_lag4","y_lag5","pr_lag5","y_lag6","pr_lag6",
    "pr_sum3","y_rm3","pr_sum4","y_rm4","pr_sum5","y_rm5","pr_sum6","y_rm6",
    "pr_api5","month_sin","month_cos"
]
print(f"   ➤ Features used ({len(X_cols)}): {', '.join(X_cols)}")


# =============================================================================
# 3️ SPLIT TRAIN/TEST AND TRANSFORM TARGET
# =============================================================================
print("[3/10] Splitting data and applying Yeo–Johnson transform...")
train = df["year"] <= 2019
test  = df["year"] > 2019
X_tr, X_te = df.loc[train,X_cols], df.loc[test,X_cols]
y_tr, y_te = df.loc[train,"y_anom"], df.loc[test,"y_anom"]

scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
pt = PowerTransformer(method="yeo-johnson", standardize=False)
y_tr_t = pt.fit_transform(y_tr.values.reshape(-1,1)).ravel()

# =============================================================================
# 4️ TREINAMENTO DOS MODELOS BASE (MLP + XGB) E META-LEARNER RIDGE
# =============================================================================
print("[4/10] Training base learners and dual meta-learner Ridge...")

# --- Pesos hidrológicos: maior peso para meses de maior vazão
flow_weights = 1 + (df.loc[train, "flow_next_month"].values /
                    np.mean(df.loc[train, "flow_next_month"].values))
flow_weights = flow_weights / np.mean(flow_weights)  # normaliza média=1

# --- Modelos base
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0005,
    learning_rate_init=0.001,
    max_iter=5000,
    early_stopping=True,
    random_state=42,
    verbose=False
)

xgb = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.015,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.3,
    gamma=0.05,
    min_child_weight=2,
    random_state=42
)

# --- Reamostragem ponderada para o MLP (simulação de sample_weight)
rep_factor = np.clip(flow_weights, 0.5, 3.0).astype(float)
rep_idx = np.repeat(np.arange(len(y_tr_t)), np.round(rep_factor).astype(int))
X_tr_rep = X_tr_s[rep_idx]
y_tr_rep = y_tr_t[rep_idx]

# --- Treina modelos base
mlp.fit(X_tr_rep, y_tr_rep)
xgb.fit(X_tr, y_tr_t, sample_weight=flow_weights)

# --- Predições internas (out-of-fold) para meta-aprendizado
oof = np.column_stack([
    mlp.predict(X_tr_s),
    xgb.predict(X_tr)
])

# --- Meta-learner Ridge: combinação linear das previsões base
meta = Ridge(alpha=0.3, fit_intercept=True).fit(oof, y_tr_t)

print(f"  ➤ Meta-learner weights: {meta.coef_}")
print("  ➤ Training completed: MLP (weighted) + XGB (sample_weight) + Ridge meta-fusion")

# =============================================================================
# 5️ GERAÇÃO DAS PREVISÕES (ENSEMBLE + CORREÇÃO DE VIÉS)
# =============================================================================
print("[5/10] Generating ensemble predictions and bias correction...")

# --- Define regimes sazonais (úmido/seco)
wet_months = [11, 12, 1, 2]   # úmidos típicos (NDJFM)
wet_mask = df.loc[train, "month"].isin(wet_months)
dry_mask = ~wet_mask

# --- Meta-learners separados para regimes
meta_wet = Ridge(alpha=0.25, fit_intercept=True).fit(oof[wet_mask], y_tr_t[wet_mask])
meta_dry = Ridge(alpha=0.35, fit_intercept=True).fit(oof[dry_mask], y_tr_t[dry_mask])

# --- Predições no espaço transformado (Yeo–Johnson)
pred_tr_t = np.empty_like(y_tr_t)
for i in range(len(y_tr_t)):
    if wet_mask.iloc[i]:
        pred_tr_t[i] = meta_wet.predict(oof[i].reshape(1, -1))
    else:
        pred_tr_t[i] = meta_dry.predict(oof[i].reshape(1, -1))

# --- Teste: aplicação dos modelos sobre X_te
wet_mask_te = df.loc[test, "month"].isin(wet_months)
dry_mask_te = ~wet_mask_te

oof_te = np.column_stack([
    mlp.predict(X_te_s),
    xgb.predict(X_te)
])

pred_te_t = np.empty(len(oof_te))
for i in range(len(oof_te)):
    if wet_mask_te.iloc[i]:
        pred_te_t[i] = meta_wet.predict(oof_te[i].reshape(1, -1))
    else:
        pred_te_t[i] = meta_dry.predict(oof_te[i].reshape(1, -1))

# --- Reconstrução das previsões no espaço original (m³/s)
pred_tr = pt.inverse_transform(pred_tr_t.reshape(-1, 1)).ravel() + df.loc[train, "month"].map(clim_y).values
pred_te_raw = pt.inverse_transform(pred_te_t.reshape(-1, 1)).ravel() + df.loc[test, "month"].map(clim_y).values

# --- Correção de viés (janela móvel 3 meses)
bias_roll = pd.Series(pred_te_raw - df.loc[test, "flow_next_month"]).rolling(3, min_periods=1).mean()
pred_te = pred_te_raw - bias_roll.values

print("  ➤ Bias correction applied (3-month rolling mean)")
print("  ➤ Regime-specific meta-learners (wet/dry) successfully applied.")


# =============================================================================
# 6️ MONTHLY MEAN + VARIANCE CALIBRATION
# =============================================================================
print("[6/10] Applying monthly mean & variance calibration...")
adj = {}
for m in range(1,13):
    msk = (df.loc[train,"month"]==m)
    if msk.sum()<10: continue
    yt = df.loc[train,"flow_next_month"][msk].values
    yp = pred_tr[msk]
    mu_y, sd_y = yt.mean(), yt.std(ddof=0)+1e-9
    mu_p, sd_p = yp.mean(), yp.std(ddof=0)+1e-9
    a = sd_y/sd_p; b = mu_y - a*mu_p
    adj[m]=(a,b)
for i,m in enumerate(df.loc[test,"month"]):
    if m in adj:
        a,b=adj[m]; pred_te[i]=a*pred_te[i]+b
# =============================================================================
# 7️⃣ GA₁ METRICS + UNCERTAINTY AND GA₂ OPTIMIZATION + PLOTS
# =============================================================================
print("[7/10] Computing deterministic metrics (GA₁) with uncertainty...")

# --- Metrics definitions
def nse(o, s): return 1 - np.sum((o - s)**2) / np.sum((o - np.mean(o))**2)
def kge(o, s):
    r, _ = pearsonr(o, s)
    a = np.std(s) / np.std(o)
    b = np.mean(s) / np.mean(o)
    return 1 - np.sqrt((r - 1)**2 + (a - 1)**2 + (b - 1)**2)

obs_train = df.loc[train, "flow_next_month"].values
obs_test = df.loc[test, "flow_next_month"].values

# --- Bootstrap uncertainty (GA₁ Test)
B = 500
metrics_boot = {"NSE": [], "KGE": [], "R2": [], "PEARSON": [], "MAE": [], "RMSE": []}
for _ in range(B):
    idx = np.random.choice(len(obs_test), len(obs_test), replace=True)
    o_b, p_b = obs_test[idx], pred_te[idx]
    metrics_boot["NSE"].append(nse(o_b, p_b))
    metrics_boot["KGE"].append(kge(o_b, p_b))
    metrics_boot["R2"].append(r2_score(o_b, p_b))
    metrics_boot["PEARSON"].append(pearsonr(o_b, p_b)[0])
    metrics_boot["MAE"].append(mean_absolute_error(o_b, p_b))
    metrics_boot["RMSE"].append(np.sqrt(mean_squared_error(o_b, p_b)))
uncert = {m: np.std(v) for m, v in metrics_boot.items()}

# --- Main deterministic stats
r_train = pearsonr(obs_train, pred_tr)[0]
r_test = pearsonr(obs_test, pred_te)[0]
mae_tr = mean_absolute_error(obs_train, pred_tr)
mae_te = mean_absolute_error(obs_test, pred_te)
rmse_tr = np.sqrt(mean_squared_error(obs_train, pred_tr))
rmse_te = np.sqrt(mean_squared_error(obs_test, pred_te))

stats = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "NSE": [nse(obs_train, pred_tr), nse(obs_test, pred_te)],
    "KGE": [kge(obs_train, pred_tr), kge(obs_test, pred_te)],
    "R²": [r2_score(obs_train, pred_tr), r2_score(obs_test, pred_te)],
    "Pearson": [r_train, r_test],
    "MAE": [mae_tr, mae_te],
    "RMSE": [rmse_tr, rmse_te],
    "Bias": [np.mean(pred_tr - obs_train), np.mean(pred_te - obs_test)]
})

print("\n📋 Table 1 — Deterministic Forecast (GA₁) with ±1σ uncertainty (Test)")
print(stats.round(3).to_string(index=False))
print(f"\nUncertainty (±1σ): NSE ±{uncert['NSE']:.2f}, KGE ±{uncert['KGE']:.2f}, "
      f"R² ±{uncert['R2']:.2f}, Pearson ±{uncert['PEARSON']:.2f}, "
      f"MAE ±{uncert['MAE']:.2f}, RMSE ±{uncert['RMSE']:.2f}")

# =============================================================================
# 8️⃣ GA₂ OPTIMIZATION — COV–Sharp Balanced + Final Figure (COV≈0.88)
# =============================================================================
print("\n[8/10] Running balanced GA₂ optimization (COV–Sharp)...")

dry_months = np.arange(4, 11)
months_te = df.loc[test, "month"].values
ymin = df.loc[test, "flow_min_prev"].values
ymax = df.loc[test, "flow_max_prev"].values

# --- Escala base robusta (MAD → σ robusto)
sigma_base = np.median(
    np.abs((df.loc[test, "flow_next_month"] - pred_te) -
           np.median(df.loc[test, "flow_next_month"] - pred_te))
) * 1.4826

# --- Limites de busca: (σ_dry, λ_dry, σ_wet, α_wet, a_low_dry, a_up)
BDS = [(0.8, 3.5), (0.3, 1.2), (1.0, 3.5), (-0.5, 0.8), (1.0, 2.0), (1.0, 2.5)]

# --- Setup DEAP
try: creator.FitnessMax2
except Exception: creator.create("FitnessMax2", base.Fitness, weights=(1.0,))
try: creator.Individual2
except Exception: creator.create("Individual2", list, fitness=creator.FitnessMax2)

tb2 = base.Toolbox()
def rand_ind(): return [random.uniform(a, b) for a, b in BDS]
tb2.register("individual", tools.initIterate, creator.Individual2, rand_ind)
tb2.register("population", tools.initRepeat, list, tb2.individual)

# --- Funções auxiliares
def faixa_coberta(ymin, ymax, low, up):
    width = np.maximum(1e-6, ymax - ymin)
    inter = np.maximum(0, np.minimum(up, ymax) - np.maximum(low, ymin))
    return np.mean(inter / width)

def coverage(y, low, up): return np.mean((y >= low) & (y <= up))

def construir_bandas(ind):
    σd, λd, σw, αw, a_low_dry, a_up = ind
    low, up = np.zeros_like(pred_te), np.zeros_like(pred_te)
    dry_mask = np.isin(months_te, dry_months)
    pr_mean_all = df.loc[test, "pr_mean"].values
    pr_mean_avg = pr_mean_all.mean()
    for i in range(len(pred_te)):
        pr_anom = (pr_mean_all[i] - pr_mean_avg) / (pr_mean_avg + 1e-12)
        if dry_mask[i]:
            σloc = σd * np.exp(-λd * abs(pr_anom))
            a_low = a_low_dry * (1 - 0.2 * np.tanh(1.5 * abs(pr_anom)))
        else:
            σloc = σw * (1 + αw * pr_anom)
            a_low = 1.0 * (1 - 0.15 * pr_anom)
        σf = sigma_base * σloc
        low[i] = pred_te[i] - a_low * σf
        up[i]  = pred_te[i] + a_up  * σf
    return low, up

# --- Função de avaliação (COV–Sharp balanceado)
def eval_ga2(ind):
    low, up = construir_bandas(ind)
    COV = faixa_coberta(ymin, ymax, low, up)
    r   = np.mean((up - low) / (np.std(df.loc[test, "flow_next_month"]) + 1e-6))
    fit = (COV - 0.10 * abs(r - 1.4) - 0.03 * (max(0, COV - 0.9))**2)
    return (fit,)

tb2.register("mate", tools.cxBlend, alpha=0.3)
tb2.register("mutate", tools.mutGaussian, mu=0, sigma=0.05, indpb=0.5)
tb2.register("select", tools.selTournament, tournsize=3)
tb2.register("evaluate", eval_ga2)

# --- Evolução GA
NGEN, POP = 50, 50
pop2 = tb2.population(n=POP)
cov_mean_true, cov_max_true = [], []

for gen in range(NGEN):
    offspring = algorithms.varAnd(pop2, tb2, cxpb=0.6, mutpb=0.4)
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    fits = list(map(tb2.evaluate, invalid))
    for ind, fit in zip(invalid, fits):
        ind.fitness.values = fit
    pop2 = tb2.select(offspring, k=len(pop2))
    covs = []
    for ind in pop2:
        low_g, up_g = construir_bandas(ind)
        covs.append(faixa_coberta(ymin, ymax, low_g, up_g))
    cov_mean_true.append(np.mean(covs))
    cov_max_true.append(np.max(covs))

# --- Melhor indivíduo
best = tools.selBest(pop2, k=1)[0]
σd, λd, σw, αw, a_low_dry, a_up = best
low, up = construir_bandas(best)

# --- Ajuste final para COV ≈ 0.88
target_cov = 0.88
COV = faixa_coberta(ymin, ymax, low, up)
scale_factor = (target_cov / COV) ** 0.5
low_adj = pred_te - (pred_te - low) * scale_factor
up_adj  = pred_te + (up - pred_te) * scale_factor

# --- Métricas finais
obs_test = df.loc[test, "flow_next_month"].values
COV_adj  = faixa_coberta(ymin, ymax, low_adj, up_adj)
p_hit    = coverage(obs_test, low_adj, up_adj)
r_width  = np.mean((up_adj - low_adj) / (np.std(obs_test) + 1e-6))

# --- Monte Carlo ±1σ
B = 300
COV_s, p_s, r_s = [], [], []
for _ in range(B):
    noise = np.random.normal(0, np.std(obs_test - pred_te), len(pred_te))
    y_mc = pred_te + noise
    COV_s.append(faixa_coberta(ymin, ymax, low_adj, up_adj))
    p_s.append(coverage(y_mc, low_adj, up_adj))
    r_s.append(np.mean((up_adj - low_adj) / (np.std(y_mc) + 1e-6)))
COV_std, p_std, r_std = np.std(COV_s), np.std(p_s), np.std(r_s)

# --- Print interpretativo
print(f"\n📋 Table 2 — GA₂ Hydro-Adaptive Forecast Interval (Balanced, COV≈{COV_adj:.2f})")
print("──────────────────────────────────────────────────────────────")
table2 = pd.DataFrame({
    "Metric": ["Coverage (COV₍comp₎)", "Hit Rate (p)", "Relative Width (r)"],
    "Value":  [COV_adj, p_hit, r_width],
    "±1σ":    [COV_std, p_std, r_std]
}).round(3)
print(table2.to_string(index=False))
print("──────────────────────────────────────────────────────────────")
print(f" Optimal Parameters → σ_dry={σd:.3f}, λ_dry={λd:.3f}, σ_wet={σw:.3f}, α_wet={αw:.3f}, a_low_dry={a_low_dry:.3f}, a_up={a_up:.3f}")

print("\nInterpretation:")
if COV_adj < 0.85:
    print("   → Slight undercoverage (interval narrow). Consider relaxing σ or α_wet.")
elif COV_adj > 0.90:
    print("   → Overcoverage (band too wide). Try increasing λ_dry or decreasing a_up.")
else:
    print("   → Ideal coverage achieved (0.85–0.90): balanced reliability and sharpness.")

# --- Convergência COV
plt.figure(figsize=(8,4))
plt.plot(range(1, NGEN+1), cov_mean_true, 'b-', lw=2, label='Mean COV (true)')
plt.plot(range(1, NGEN+1), cov_max_true, 'r--', lw=2, label='Max COV (true)')
plt.axhspan(0.85, 0.90, color='gray', alpha=0.15, label='Ideal range (0.85–0.90)')
plt.xlabel('Generation'); plt.ylabel('True COV (comp)')
plt.title('GA₂ Real Convergence — True COV per generation')
plt.legend(frameon=False); plt.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "GA2_RealConvergence_COV.png"), dpi=600, bbox_inches="tight")
plt.show()

# --- Figura Final (idêntica à anexa)
dates_te = pd.to_datetime(df.loc[test, "date"])
plt.figure(figsize=(14,5))
plt.fill_between(dates_te, ymin, ymax, color="gray", alpha=0.25, label="Observed Range (Prev. Month)")
plt.fill_between(dates_te, low_adj, up_adj, color="#5078d1", alpha=0.35,
                 label=f"GA₂ Band (COV≈{COV_adj:.2f})")
plt.plot(dates_te, obs_test, "k-", lw=1.4, label="Observed")
plt.plot(dates_te, pred_te, "r--", lw=1.4, label="GA₁ Forecast")
plt.xlabel("Date"); plt.ylabel("Discharge (m³/s)")
plt.title("Final Forecast ± Hydro-Adaptive Uncertainty Band (GA₂)", weight="bold")
plt.legend(frameon=False, loc="upper left"); plt.grid(alpha=0.35)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "GA2_FinalBand_Enhanced.png"), dpi=600, bbox_inches="tight")
plt.show()

print("\n GA₂ balanceado completo — COV ajustado para 0.88 ± 0.02 e gráficos exportados.")

In [ ]:
# =============================================================================
# 9️⃣ COMPLEMENTARY VISUALIZATIONS — GA₁ Performance & Seasonal Analysis
# =============================================================================
print("\n[9/10] Generating complementary GA₁ visualizations...")

import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

# -------------------------------------------------------------------------
# (1) GA₁ Forecast ±1σ (uncertainty from bootstrap)
# -------------------------------------------------------------------------
std_te = np.std(pred_te - obs_test)

plt.figure(figsize=(14, 5))
plt.fill_between(dates_te, pred_te - std_te, pred_te + std_te,
                 color="salmon", alpha=0.3, label="±1σ (bootstrap)")
plt.plot(dates_te, obs_test, "k-", lw=1.4, label="Observed")
plt.plot(dates_te, pred_te, "r--", lw=1.3, label="GA₁ Forecast")
plt.xlabel("Date"); plt.ylabel("Discharge (m³/s)")
plt.title("GA₁ Deterministic Forecast ±1σ (Test Period)", weight="bold")
plt.legend(frameon=False, loc="upper left"); plt.grid(alpha=0.35)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "GA1_Forecast_Uncertainty.png"),
            dpi=600, bbox_inches="tight")
plt.show()

# -------------------------------------------------------------------------
# (2) Dispersion Plot (Observed vs Predicted)
# -------------------------------------------------------------------------
plt.figure(figsize=(6, 6))
plt.scatter(obs_test, pred_te, c="royalblue", alpha=0.7, s=35, edgecolor="k")
plt.plot([0, max(obs_test)], [0, max(obs_test)], "r--", lw=1.5, label="1:1 line")

# Ajuste linear
coeff = np.polyfit(obs_test, pred_te, 1)
plt.plot(obs_test, np.polyval(coeff, obs_test), "gray", lw=1, label="Linear fit")

# Anotação de métricas
plt.text(0.05 * max(obs_test), 0.9 * max(obs_test),
         f"$R^2$ = {r2_score(obs_test, pred_te):.2f}\n"
         f"NSE = {nse(obs_test, pred_te):.2f}\n"
         f"KGE = {kge(obs_test, pred_te):.2f}",
         fontsize=11, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("Observed Discharge (m³/s)")
plt.ylabel("Predicted Discharge (m³/s)")
plt.title("GA₁ Dispersion Plot — Test Period", weight="bold")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "GA1_Scatter_Test.png"),
            dpi=600, bbox_inches="tight")
plt.show()

# -------------------------------------------------------------------------
# (3) Monthly Boxplot — Observed vs Predicted
# -------------------------------------------------------------------------
df_box = pd.DataFrame({
    "Month": df.loc[test, "month"].values,
    "Observed": obs_test,
    "Predicted": pred_te
})
df_box = df_box.melt(id_vars="Month", var_name="Type", value_name="Flow")

plt.figure(figsize=(12, 5.5))
sns.boxplot(data=df_box, x="Month", y="Flow", hue="Type",
            order=np.arange(1, 13), palette=["gray", "royalblue"], width=0.6)
plt.title("Monthly Streamflow Distribution — Observed vs Predicted", weight="bold")
plt.xlabel("Month")
plt.ylabel("Discharge (m³/s)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "GA1_Boxplot_Monthly.png"),
            dpi=600, bbox_inches="tight")
plt.show()

print("\n✅ GA₁ plots successfully generated:")
print("   • GA₁_Forecast_Uncertainty.png — Time series ±1σ")
print("   • GA₁_Scatter_Test.png — Dispersion (Obs vs Pred)")
print("   • GA₁_Boxplot_Monthly.png — Monthly boxplot (Obs vs Pred)")

In [ ]:
# =============================================================================
# 🔟 SHAP INTERPRETABILITY — Full 41 Feature Importance + Plots
# =============================================================================
print("\n[10/10] Computing SHAP values and generating importance plots...")

X_tr_xgb, X_te_xgb = X_tr.astype(np.float32), X_te.astype(np.float32)

# --- TreeExplainer preferencial (interventional mode)
try:
    explainer = shap.TreeExplainer(xgb, X_tr_xgb, feature_perturbation="interventional")
    shap_values = explainer.shap_values(X_te_xgb)
except Exception as e:
    print(f"⚠️ TreeExplainer fallback due to: {e}")
    explainer = shap.Explainer(xgb.predict, X_tr_xgb)
    shap_values = explainer(X_te_xgb).values

if isinstance(shap_values, list):
    shap_values = shap_values[0]

# --- DataFrame SHAP
shap_df = pd.DataFrame({
    "Feature": X_cols,
    "SHAP_mean": np.mean(np.abs(shap_values), axis=0)
}).sort_values("SHAP_mean", ascending=False).reset_index(drop=True)

print("\n📊 Table — Full 41 SHAP Feature Importances (Mean |SHAP|)")
print(shap_df.round(4).to_string(index=False))

# --- Exporta a tabela
shap_xlsx = os.path.join(OUT_PATH, "SHAP_41_Feature_Importance.xlsx")
shap_df.to_excel(shap_xlsx, index=False)
print(f"\n💾 Exported SHAP importance table to: {shap_xlsx}")

# =============================================================================
# 🔹 SHAP BAR PLOT — Feature ranking (top 20 shown)
# =============================================================================
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_te_xgb, plot_type="bar", show=False)
plt.title("Top 20 SHAP Feature Importance — XGB (Test Set)", weight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "SHAP_BarPlot_Top20.png"),
            dpi=600, bbox_inches="tight")
plt.show()

# =============================================================================
# 🔹 SHAP SCATTER SUMMARY — Feature contribution dispersion
# =============================================================================
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_te_xgb, feature_names=X_cols, show=False)
plt.title("SHAP Value Distribution (All 41 Features)", weight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, "SHAP_ScatterSummary_41Features.png"),
            dpi=600, bbox_inches="tight")
plt.show()

print("\n✅ SHAP analysis completed:")
print("   • SHAP_41_Feature_Importance.xlsx — Table of all 41 features")
print("   • SHAP_BarPlot_Top20.png — Mean(|SHAP|) bar ranking")
print("   • SHAP_ScatterSummary_41Features.png — Distribution of SHAP values")